## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


Reproducibility in this course will be achieved by replacing the authors’ analysis with nf-core pipelines

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

Check how good the raw reads are, trim off the noise, map them to the genome, count the reads per gene, and then put it all into a gene-by-sample table.<br>
The nf-core pipeline doing this is: rnaseq

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

Takes your raw FASTQ files and runs them through a whole bunch of steps to give you a count table, plus it checks the quality along the way.<br>

The tools used by the pipeline are the followings:
FastQC,
MultiQC,
Trim Galore (or fastp),
STAR (or HISAT2),
Salmon (or Kallisto),
SAMtools,
Picard,
featureCounts,
DESeq2

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Carica metadata e SRR scaricati
meta = pd.read_csv("metadata_long.csv", dtype={"sample": str})
srr_ids = pd.read_csv("srr_ids.csv", header=None, dtype=str)[0].str.strip().tolist()

# 2. Filtra metadata ai soli campioni di interesse
meta = meta[meta["sample"].isin(srr_ids)].reset_index(drop=True)

# 3. Trova FASTQ (assumendo naming convention standard)
fastq_dir = Path("results/fetchngs/fastq")
def find_fastq(srr, read):
    matches = list(fastq_dir.glob(f"*{srr}*{read}.fastq.gz"))
    if not matches:
        raise FileNotFoundError(f"{read} FASTQ non trovato per {srr}")
    return str(matches[0])

# 4. Costruisci samplesheet
rows = []
for _, r in meta.iterrows():
    srr = r["sample"]
    rows.append({
        "sample":       f"{r['genotype']}_{r['condition']}_{srr}",
        "fastq_1":      find_fastq(srr, "_1"),
        "fastq_2":      find_fastq(srr, "_2"),
        "strandedness": "reverse",
    })

samplesheet = pd.DataFrame(rows)
samplesheet.to_csv("samplesheet.csv", index=False)

# 5. Verifica esistenza file
missing = []
for _, row in samplesheet.iterrows():
    for col in ("fastq_1", "fastq_2"):
        if not Path(row[col]).is_file():
            missing.append(row[col])

if missing:
    raise FileNotFoundError(f"File mancanti: {missing}")

print("Samplesheet creata con successo:")
print(samplesheet.to_string(index=False))

        sample condition genotype     n_bases
0  SRR23195511       Oxy     Sham  6456390900
1  SRR23195516       Oxy      SNI  6203117700

Samplesheet:
              sample                                                   fastq_1                                                   fastq_2 strandedness
Sham_Oxy_SRR23195511 results/fetchngs/fastq/SRX19144488_SRR23195511_1.fastq.gz results/fetchngs/fastq/SRX19144488_SRR23195511_2.fastq.gz      reverse
 SNI_Oxy_SRR23195516 results/fetchngs/fastq/SRX19144486_SRR23195516_1.fastq.gz results/fetchngs/fastq/SRX19144486_SRR23195516_2.fastq.gz      reverse
  ✅ results/fetchngs/fastq/SRX19144488_SRR23195511_1.fastq.gz
  ✅ results/fetchngs/fastq/SRX19144488_SRR23195511_2.fastq.gz
  ✅ results/fetchngs/fastq/SRX19144486_SRR23195516_1.fastq.gz
  ✅ results/fetchngs/fastq/SRX19144486_SRR23195516_2.fastq.gz


In [22]:
!nextflow run nf-core/rnaseq \
    -profile docker \
    -c mycustom.config \
    --input samplesheet.csv \
    --outdir results/rnaseq \
    --genome GRCm38 \
    --aligner hisat2 \
    -resume \
    -ansi-log false

N E X T F L O W  ~  version 26.04.6
Launching `https://github.com/nf-core/rnaseq` [stoic_gilbert] - revision: a1fcdddd3b [master]
WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/rnaseq 3.27.0
------------------------------------------------------

Input/output options
  input              : samplesheet.csv
  outdir             : results/rnaseq

Reference genome options
  genome             : GRCm38
  fasta              : s3://ngi-igenomes/igenomes//Mus_musculus/Ensembl/GRCm38/Sequence/WholeGenomeFasta/genome.fa
  gtf                : s3://ngi-igenomes/igenomes//Mus_musculus/Ensembl/GRCm38/Annotation/Genes/genes.gtf
  gene_bed           : s3://ngi-igenomes/igenomes//Mus_musculu

Explain all the parameters you set and why you set them in this way.



--input samplesheet.csv: The list of samples and their FASTQ files.<br>
--outdir results/rnaseq: Where the pipeline writes its outputs.<br>
--genome GRCm38: Tells the pipeline to use a pre-built iGenomes reference for the mouse genome GRCm38. This avoids having to download and index the genome manually<br>
--aligner hisat2 chosen over STAR for lower memory footprint

How did the pipeline perform?

The pipeline appears to have completed successfully, MultiQC aggregated all expected nf-core/rnaseq QC modules 

In [40]:
# Programmatically extract pipeline performance metrics from MultiQC
import subprocess
import pandas as pd

print("This table shows pipeline performance metrics from your nf-core/rnaseq run ")

DATA = '/Users/tesbaro/Desktop/CompWorkflow/computational-workflows-2026/day_02/results/rnaseq/multiqc/hisat2/multiqc_report_data/multiqc_general_stats.txt'

awk_cmd = '''
$1 == "SNI_Oxy_SRR23195516" {
    print "SNI_Oxy\tAlignment\t" $3
    print "SNI_Oxy\tReadsMapped_M\t" $13
    print "SNI_Oxy\tMappedPct\t" $14
    print "SNI_Oxy\tProperPairs\t" $10
    print "SNI_Oxy\tErrorRate\t" $11
    print "SNI_Oxy\tExonic\t" $6
    print "SNI_Oxy\tIntronic\t" $7
    print "SNI_Oxy\tDuplication\t" $33
    print "SNI_Oxy\tInsertSize\t" $18
}
$1 == "Sham_Oxy_SRR23195511_1" {
    print "Sham_Oxy\tAlignment\t" $29
    print "Sham_Oxy\tErrorRate\t" $30
}
$1 == "Sham_Oxy_SRR23195511" {
    print "Sham_Oxy\tExonic\t" $28
    print "Sham_Oxy\tIntronic\t" $30
    print "Sham_Oxy\tDuplication\t" $37
    print "Sham_Oxy\tReadsMapped_M\t" ($27 * $29 / 100)
    print "Sham_Oxy\tMappedPct\t" $29
    print "Sham_Oxy\tProperPairs\t" $29
    print "Sham_Oxy\tInsertSize\t" $36
}
'''

# Run awk extraction
result = subprocess.run(['awk', '-F\t', awk_cmd, DATA], capture_output=True, text=True)

# Parse into DataFrame
metrics = {}
for line in result.stdout.strip().split('\n'):
    sample, metric, value = line.split('\t')
    metrics.setdefault(sample, {})[metric] = float(value)

df = pd.DataFrame(metrics).T
df.columns.name = 'Sample'

# Display formatted (styled only - no duplicate print)
df_display = df.round(2)
cols_order = ['Alignment', 'ReadsMapped_M', 'MappedPct', 'ProperPairs', 
              'ErrorRate', 'Exonic', 'Intronic', 'Duplication', 'InsertSize']
df_display = df_display[cols_order]

# Style for notebook
styled = df_display.style.format(precision=2)
display(styled)

This table shows pipeline performance metrics from your nf-core/rnaseq run 


Sample,Alignment,ReadsMapped_M,MappedPct,ProperPairs,ErrorRate,Exonic,Intronic,Duplication,InsertSize
SNI_Oxy,85.48,34.09,100.00,90.28,0.36,17.57,11.17,58.32,517.10
Sham_Oxy,99.40,42.79,99.40,99.40,0.42,57.15,1.67,9.09,147.00


Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication
- GC content

What are the possible steps that could lead to poorer results?

Check: read quality, adapters, duplication, GC, mapping, strandedness<br>

rRNA: high rRNA reduces usable reads.<br>
Duplication: high (e.g. SRX19144489 88%, SRX19144481 81%) = low complexity.<br>
GC: normal ~47–52%. SRX19144492 is an outlier (63–68%).<br>
Poor results from: bad RNA, rRNA, adapters, over-trimming, GC bias, wrong reference/strandedness.<br>

Remember: poor results can come from bad input, rRNA, adapter dimers, over-trimming, wrong reference, or wrong strandedness.

Would you exclude any samples? If yes, which and why?

yes, I would exclude a sample if it has extreme GC content, low complexity, or very few expressed genes, because it would be an outlier. <br>
SRX19144489 high duplication and error rate, failed strandedness<br>
SRX19144481, 84: also failed strandedness<br>
SRX19144492: very low alignment, extreme GC<br>

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

differential analisys of gene expression in the groups: SNI_Oxy vs Sham_Oxy

R: DESeq2, edgeR, limma, tximport, clusterProfiler, ggplot2.<br>
Python: PyDESeq2, pandas, scipy, seaborn, gseapy.